In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Stock Price Prediction - Exploratory Data Analysis\n",
    "\n",
    "This notebook explores the stock price data and experiments with different approaches.\n",
    "\n",
    "## Table of Contents\n",
    "1. [Data Loading](#data-loading)\n",
    "2. [Data Exploration](#data-exploration)\n",
    "3. [Feature Engineering](#feature-engineering)\n",
    "4. [Model Training](#model-training)\n",
    "5. [Model Evaluation](#model-evaluation)\n",
    "6. [Predictions](#predictions)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import sys\n",
    "import os\n",
    "sys.path.append(os.path.dirname(os.getcwd()))\n",
    "\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Import custom modules\n",
    "from src.data_loader import load_stock_data, get_data_info\n",
    "from src.preprocess import preprocess_data\n",
    "from src.train_model import train_all_models, prepare_features_target, split_data\n",
    "from src.evaluate import evaluate_all_models\n",
    "from src.visualization import *\n",
    "from utils.config import STOCK_SYMBOL\n",
    "\n",
    "# Set display options\n",
    "pd.set_option('display.max_columns', None)\n",
    "plt.style.use('seaborn-v0_8')\n",
    "%matplotlib inline"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Data Loading <a id='data-loading'></a>\n",
    "\n",
    "Load stock data from Yahoo Finance"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load data\n",
    "print(f\"Loading {STOCK_SYMBOL} stock data...\")\n",
    "df = load_stock_data(STOCK_SYMBOL, force_download=False)\n",
    "\n",
    "# Display basic info\n",
    "get_data_info(df)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Display first few rows\n",
    "df.head(10)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Display last few rows\n",
    "df.tail(10)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Data Exploration <a id='data-exploration'></a>\n",
    "\n",
    "Visualize and understand the data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot price history\n",
    "plot_price_history(df, save=False)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot OHLC data\n",
    "plot_ohlc_data(df, last_n_days=100, save=False)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot trading volume\n",
    "plot_volume(df, save=False)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Statistical summary\n",
    "df[['Open', 'High', 'Low', 'Close', 'Volume']].describe()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Check for missing values\n",
    "df.isnull().sum()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Feature Engineering <a id='feature-engineering'></a>\n",
    "\n",
    "Create features for machine learning models"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Preprocess data and create features\n",
    "df_processed = preprocess_data(df, save=False)\n",
    "\n",
    "print(f\"\\nProcessed data shape: {df_processed.shape}\")\n",
    "print(f\"\\nNew columns created: {set(df_processed.columns) - set(df.columns)}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Display processed data\n",
    "df_processed.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot moving averages\n",
    "plot_moving_averages(df_processed, save=False)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot correlation matrix\n",
    "plot_correlation_matrix(df_processed, save=False)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Analyze price changes\n",
    "if 'price_change_pct' in df_processed.columns:\n",
    "    plt.figure(figsize=(12, 4))\n",
    "    plt.hist(df_processed['price_change_pct'].dropna(), bins=50, alpha=0.7, edgecolor='black')\n",
    "    plt.axvline(x=0, color='r', linestyle='--', linewidth=2)\n",
    "    plt.title('Distribution of Daily Returns (%)')\n",
    "    plt.xlabel('Return (%)')\n",
    "    plt.ylabel('Frequency')\n",
    "    plt.grid(True, alpha=0.3)\n",
    "    plt.show()\n",
    "    \n",
    "    print(f\"Mean daily return: {df_processed['price_change_pct'].mean():.3f}%\")\n",
    "    print(f\"Std daily return: {df_processed['price_change_pct'].std():.3f}%\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. Model Training <a id='model-training'></a>\n",
    "\n",
    "Train multiple machine learning models"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Prepare features and target\n",
    "X, y, feature_cols = prepare_features_target(df_processed)\n",
    "\n",
    "print(f\"\\nFeatures shape: {X.shape}\")\n",
    "print(f\"Target shape: {y.shape}\")\n",
    "print(f\"\\nNumber of features: {len(feature_cols)}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Split data\n",
    "X_train, X_test, y_train, y_test = split_data(X, y)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Train all models\n",
    "models = train_all_models(X_train, y_train, save_models=False)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. Model Evaluation <a id='model-evaluation'></a>\n",
    "\n",
    "Evaluate and compare model performance"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Evaluate all models\n",
    "results = evaluate_all_models(models, X_train, y_train, X_test, y_test, save=False)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot predictions vs actual for XGBoost\n",
    "if 'xgboost' in models:\n",
    "    y_pred = models['xgboost'].predict(X_test)\n",
    "    plot_predictions_vs_actual(y_test, y_pred, title='XGBoost Predictions', save=False)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot residuals\n",
    "if 'xgboost' in models:\n",
    "    y_pred = models['xgboost'].predict(X_test)\n",
    "    plot_residuals(y_test, y_pred, save=False)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Feature importance\n",
    "from src.train_model import get_feature_importance\n",
    "\n",
    "if 'xgboost' in models:\n",
    "    importance_df = get_feature_importance(models['xgboost'], feature_cols, top_n=20)\n",
    "    plot_feature_importance(importance_df, top_n=15, save=False)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. Predictions <a id='predictions'></a>\n",
    "\n",
    "Make future predictions"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Simple prediction example\n",
    "best_model = models['xgboost']\n",
    "\n",
    "# Predict on test set\n",
    "test_predictions = best_model.predict(X_test)\n",
    "\n",
    "# Create results DataFrame\n",
    "results_df = pd.DataFrame({\n",
    "    'Actual': y_test.values,\n",
    "    'Predicted': test_predictions,\n",
    "    'Difference': y_test.values - test_predictions,\n",
    "    'Percent_Error': ((y_test.values - test_predictions) / y_test.values * 100)\n",
    "})\n",
    "\n",
    "print(\"\\nSample Predictions:\")\n",
    "print(results_df.head(10))\n",
    "print(f\"\\nMean Absolute Percent Error: {abs(results_df['Percent_Error']).mean():.2f}%\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Experiments Section\n",
    "\n",
    "Try your own experiments below:"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Your experiments here\n"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Conclusions\n",
    "\n",
    "Write your conclusions here:\n",
    "\n",
    "1. **Best Model**: \n",
    "2. **Key Features**: \n",
    "3. **Limitations**: \n",
    "4. **Future Improvements**: "
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}